In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

##### 1. Load the actor-by-genre dataset


In [8]:
csv_path = Path(
    "/Users/rgaffney/Documents/UMD/Classes/Summer II/2026/"
    "INST414/Data Files/imdb_movies_2000to2022.actorXgenre.csv"
)

if not csv_path.exists():
    raise FileNotFoundError(
        f"Could not find the data file:\n{csv_path}"
    )

actor_genre_df = pd.read_csv(
    csv_path,
    index_col="actor_id"
)

print("Dataset loaded successfully.")
print("Shape:", actor_genre_df.shape)
display(actor_genre_df.head())


Dataset loaded successfully.
Shape: (33609, 25)


,Comedy,Fantasy,Romance,Drama,Mystery,Thriller,Action,Biography,Crime,War,...,Horror,Documentary,Sport,News,Family,Music,Unnamed: 22,Western,Short,Reality-TV
actor_id,,,,,,,,,,,,,,,,,,,,,
nm0000212,7.0,1.0,6.0,6.0,1.0,2.0,1.0,1.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0413168,7.0,3.0,5.0,12.0,5.0,2.0,14.0,4.0,6.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0000630,8.0,2.0,6.0,14.0,2.0,3.0,4.0,5.0,1.0,1.0,...,3.0,7.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0005227,10.0,1.0,2.0,2.0,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
nm0864851,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### 2. Select the numeric genre columns


In [9]:
actor_genre_numeric_df = (
    actor_genre_df
    .select_dtypes(include="number")
    .fillna(0)
)

if actor_genre_numeric_df.empty:
    raise ValueError(
        "No numeric genre columns were found in the dataset."
    )

print("Number of genre features:", actor_genre_numeric_df.shape[1])
print("Genre columns:")
print(actor_genre_numeric_df.columns.tolist())

Number of genre features: 25
Genre columns:
['Comedy', 'Fantasy', 'Romance', 'Drama', 'Mystery', 'Thriller', 'Action', 'Biography', 'Crime', 'War', 'Adventure', 'Sci-Fi', 'Animation', 'Musical', 'History', 'Horror', 'Documentary', 'Sport', 'News', 'Family', 'Music', 'Unnamed: 22', 'Western', 'Short', 'Reality-TV']


#### 3. Apply L1 normalization

In [10]:
actor_genre_l1_df = pd.DataFrame(
    normalize(
        actor_genre_numeric_df,
        norm="l1"
    ),
    index=actor_genre_numeric_df.index,
    columns=actor_genre_numeric_df.columns
)

print("\nNormalized data:")
display(actor_genre_l1_df.head())

print(
    "Example normalized row sum:",
    actor_genre_l1_df.iloc[0].sum()
)


Normalized data:


,Comedy,Fantasy,Romance,Drama,Mystery,Thriller,Action,Biography,Crime,War,...,Horror,Documentary,Sport,News,Family,Music,Unnamed: 22,Western,Short,Reality-TV
actor_id,,,,,,,,,,,,,,,,,,,,,
nm0000212,0.250000,0.035714,0.214286,0.214286,0.035714,0.071429,0.035714,0.035714,0.071429,0.035714,...,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0
nm0413168,0.081395,0.034884,0.058140,0.139535,0.058140,0.023256,0.162791,0.046512,0.069767,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0
nm0000630,0.112676,0.028169,0.084507,0.197183,0.028169,0.042254,0.056338,0.070423,0.014085,0.014085,...,0.042254,0.098592,0.042254,0.014085,0.00,0.0,0.0,0.0,0.0,0.0
nm0005227,0.400000,0.040000,0.080000,0.080000,0.000000,0.040000,0.040000,0.000000,0.000000,0.000000,...,0.040000,0.000000,0.040000,0.000000,0.08,0.0,0.0,0.0,0.0,0.0
nm0864851,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.333333,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0


Example normalized row sum: 1.0


#### 4. Fit the final K-Means model

In [11]:
k = 10

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20
)

actor_cluster_labels = kmeans.fit_predict(
    actor_genre_l1_df
)

print("\nK-Means model fitted.")
print("Number of actors clustered:", len(actor_cluster_labels))
print("Number of clusters:", kmeans.n_clusters)


K-Means model fitted.
Number of actors clustered: 33609
Number of clusters: 10


#### 5. Store each actor's cluster assignment

In [ ]:
cluster_results_df = pd.DataFrame(
    {
        "actor_id": actor_genre_l1_df.index,
        "cluster": actor_cluster_labels
    }
).set_index("actor_id")

display(cluster_results_df.head())

#### 6. Calculate distances to the cluster centroids

In [12]:
distances_to_centroids = kmeans.transform(
    actor_genre_l1_df
)

#### 7. Create the cluster summary table

In [13]:
summary_rows = []

for cluster_id in range(kmeans.n_clusters):

    # Positions of actors assigned to this cluster
    member_positions = np.where(
        actor_cluster_labels == cluster_id
    )[0]

    cluster_size = len(member_positions)

    # Get this cluster's centroid
    centroid = kmeans.cluster_centers_[cluster_id]

    # Find the three genres with the largest centroid values
    top_genre_positions = np.argsort(
        centroid
    )[::-1][:3]

    top_genres = actor_genre_l1_df.columns[
        top_genre_positions
    ].tolist()

    # Find the two actors closest to this centroid
    member_distances = distances_to_centroids[
        member_positions,
        cluster_id
    ]

    representative_positions = member_positions[
        np.argsort(member_distances)[:2]
    ]

    representative_actor_ids = actor_genre_l1_df.index[
        representative_positions
    ].tolist()

    summary_rows.append(
        {
            "cluster": cluster_id,
            "actor_count": cluster_size,
            "top_genre_1": top_genres[0],
            "top_genre_2": top_genres[1],
            "top_genre_3": top_genres[2],
            "representative_actor_1": representative_actor_ids[0],
            "representative_actor_2": representative_actor_ids[1]
        }
    )

cluster_summary_df = pd.DataFrame(summary_rows)

print("\nFinal cluster summary:")
display(cluster_summary_df)


Final cluster summary:


,cluster,actor_count,top_genre_1,top_genre_2,top_genre_3,representative_actor_1,representative_actor_2
0,0,6109,Comedy,Drama,Romance,nm0356021,nm0000111
1,1,1713,Documentary,Family,Animation,nm0001975,nm0066098
2,2,3965,Drama,Comedy,Thriller,nm0001583,nm0722632
3,3,4890,Action,Drama,Sci-Fi,nm0199939,nm0707425
4,4,7819,Drama,Romance,Thriller,nm0001599,nm0000677
5,5,4181,Horror,Thriller,Mystery,nm1577637,nm0878597
6,6,1997,Comedy,Horror,Drama,nm0377888,nm0714634
7,7,1606,Horror,Comedy,Drama,nm0452627,nm4066879
8,8,960,Thriller,Drama,Action,nm0906617,nm4096851
9,9,369,Unnamed: 22,Comedy,War,nm0317570,nm1368204
